In [38]:
# Standard Library
import re
import urllib.error
import urllib.request
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualisierung
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Jupyter
import ipywidgets as widgets
from IPython.display import display

# Hockey rink
from hockey_rink import NHLRink

# Setup
rink = NHLRink()

pd.options.display.max_columns = None

In [2]:
#requirements
#%pip install -U ipywidgets jupyterlab_widgets

In [51]:
# Daten laden -> Pfad Anpassen
data_dir = Path("../hockey-dataset/data")
df = pd.read_csv(data_dir / "pp_3.csv")

In [52]:
# Koordinaten von Metern in Feet umrechnen
M_TO_FT = 3.28084


def coordinates_to_feet(series):
    coords = (
        series
        .astype("string")
        .str.split(",", n=1, expand=True)
        .reindex(columns=[0, 1])
    )

    x = pd.to_numeric(coords[0], errors="coerce") * M_TO_FT
    y = pd.to_numeric(coords[1], errors="coerce") * M_TO_FT

    return x, y

In [53]:
df["EventStartX_ft"], df["EventStartY_ft"] = coordinates_to_feet(df["EventStartCoordinate"])
df["EventEndX_ft"], df["EventEndY_ft"] = coordinates_to_feet(df["EventEndCoordinate"])

In [54]:
# Spielerkoordinaten in Feet umrechnen
for i in range(1, 13):
    coord_col = f"StartPlayerCoordinates{i}"
    if coord_col not in df.columns:
        continue
    x, y = coordinates_to_feet(df[coord_col])

    df[f"StartPlayerX{i}_ft"] = x
    df[f"StartPlayerY{i}_ft"] = y

In [55]:
structure_df = df.reset_index(drop=True).copy()

In [56]:
# Spielernummer aus Namen wie "#11 Danny Nelson" extrahieren
def extract_player_number(name):
    if pd.isna(name):
        return ""

    match = re.match(r"#(\d+)", str(name).strip())

    return match.group(1) if match else str(name)

In [57]:
# Position der Torlinie in Feet
GOAL_X_FT = 89

def extract_player_number(name):
    """Extrahiert die Spielernummer aus Namen wie '#11 Danny Nelson'."""
    if pd.isna(name):
        return ""

    match = re.match(r"#(\d+)", str(name).strip())
    return match.group(1) if match else str(name)


def get_powerplay_teams(strength_type):
    if strength_type == "HomePowerplay":
        return "Home", "Away"

    if strength_type == "AwayPowerplay":
        return "Away", "Home"

    return None, None


def get_visible_players(row, attacking_team, defending_team, ax):
    players = []

    x_min, x_max = sorted(ax.get_xlim())
    y_min, y_max = sorted(ax.get_ylim())

    for i in range(1, 13):

        team = row.get(f"StartPlayerTeam{i}")
        name = row.get(f"StartPlayerName{i}")
        x = row.get(f"StartPlayerX{i}_ft")
        y = row.get(f"StartPlayerY{i}_ft")

        if (
            pd.isna(team)
            or pd.isna(name)
            or pd.isna(x)
            or pd.isna(y)
        ):
            continue

        if team not in [attacking_team, defending_team]:
            continue

        # Koordinaten an Rink-Rotation anpassen
        plot_x, plot_y = rink.convert_xy(
            [x],
            [y],
            ax=ax
        )

        plot_x = plot_x[0]
        plot_y = plot_y[0]

        # Spieler ausserhalb des sichtbaren Ausschnitts ignorieren
        if not (
            x_min <= plot_x <= x_max
            and y_min <= plot_y <= y_max
        ):
            continue

        players.append({
            "team": team,
            "name": name,
            "number": extract_player_number(name),
            "x": x,
            "y": y,
            "plot_x": plot_x,
            "plot_y": plot_y
        })

    return players


def get_defending_skaters(players, defending_team):
    defenders = [
        player
        for player in players
        if player["team"] == defending_team
    ]

    # Bei 5 Defending-Spielern:
    # Spieler mit geringster Distanz zum Tor = Goalie
    if len(defenders) >= 5:

        goalie = min(
            defenders,
            key=lambda player: min(
                np.hypot(player["x"] - GOAL_X_FT, player["y"]),
                np.hypot(player["x"] + GOAL_X_FT, player["y"])
            )
        )

        return [
            player
            for player in defenders
            if player is not goalie
        ]

    return defenders


def draw_defensive_structure(ax, defenders):
    if len(defenders) != 4:
        return

    center_x = np.mean([player["plot_x"] for player in defenders])
    center_y = np.mean([player["plot_y"] for player in defenders])

    # Spieler geometrisch um den Mittelpunkt sortieren,
    # damit sich die Polygonlinien nicht kreuzen.
    defenders_sorted = sorted(
        defenders,
        key=lambda player: np.arctan2(
            player["plot_y"] - center_y,
            player["plot_x"] - center_x
        )
    )

    polygon_coordinates = [
        (player["plot_x"], player["plot_y"])
        for player in defenders_sorted
    ]

    polygon = Polygon(
        polygon_coordinates,
        closed=True,
        facecolor="dodgerblue",
        edgecolor="dodgerblue",
        alpha=0.15,
        linewidth=2.5,
        zorder=3
    )

    ax.add_patch(polygon)


def draw_event(row, event_df, ax):
    if row["EventType"] == "PuckControl":

        rink.wavy_arrow(
            data=event_df,
            x="EventStartX_ft",
            y="EventStartY_ft",
            x2="EventEndX_ft",
            y2="EventEndY_ft",
            head_width=4,
            length_includes_head=True,
            ax=ax
        )

    elif row["EventType"] == "Pass":
        rink.arrow(
            data=event_df,
            x="EventStartX_ft",
            y="EventStartY_ft",
            x2="EventEndX_ft",
            y2="EventEndY_ft",
            head_width=0,
            length_includes_head=True,
            ax=ax
        )
    elif row["EventType"] == "Shot":
        rink.scatter(
            data=event_df,
            x="EventStartX_ft",
            y="EventStartY_ft",
            s=220,
            marker="*",
            color="gold",
            edgecolors="black",
            zorder=6,
            ax=ax
        )


def plot_frame(frame):

    row = structure_df.iloc[frame]
    # Powerplay-Teams bestimmen
    attacking_team, defending_team = get_powerplay_teams(
        row["strength_type_state"]
    )

    # Rink zeichnen
    fig, ax = plt.subplots(figsize=(10, 8))
    rink.draw(
        display_range="defence",
        rotation=90,
        ax=ax
    )

    # Aktuelles Event
    event_df = structure_df.iloc[[frame]]

    draw_event(
        row=row,
        event_df=event_df,
        ax=ax
    )

    # Spieler sammeln
    players = get_visible_players(
        row=row,
        attacking_team=attacking_team,
        defending_team=defending_team,
        ax=ax
    )

    # PK-Skater ohne Goalie
    defending_skaters = get_defending_skaters(
        players=players,
        defending_team=defending_team
    )

    # Defensive Struktur zeichnen
    draw_defensive_structure(
        ax=ax,
        defenders=defending_skaters
    )

    # Legende
    legend_elements = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="None",
            label="Powerplay Team",
            markerfacecolor="crimson",
            markeredgecolor="black",
            markersize=10
        ),
        Line2D(
            [0], [0],
            marker="o",
            linestyle="None",
            label="Defending Team",
            markerfacecolor="dodgerblue",
            markeredgecolor="black",
            markersize=10
        ),
        Line2D(
            [0], [0],
            marker="*",
            linestyle="None",
            label="Shot",
            markerfacecolor="gold",
            markeredgecolor="black",
            markersize=14
        )
    ]

    # Spieler zeichnen
    for player in players:

        facecolor = (
            "crimson"
            if player["team"] == attacking_team
            else "dodgerblue"
        )

        ax.text(
            player["plot_x"],
            player["plot_y"],
            player["number"],
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
            color="white",
            bbox=dict(
                boxstyle="circle,pad=0.35",
                facecolor=facecolor,
                edgecolor="black",
                linewidth=1.5
            ),
            zorder=5
        )

        legend_elements.append(
            Line2D(
                [0], [0],
                marker="o",
                linestyle="None",
                label=player["name"],
                markerfacecolor=facecolor,
                markeredgecolor="black",
                markersize=9
            )
        )

    # Titel
    ax.set_title(
        f'Event {frame + 1} / {len(structure_df)} | '
        f'Period {row["Period"]} | '
        f'Clock {row["MatchClock"]} | '
        f'{row["EventType"]}'
    )

    # Legende
    ax.legend(
        handles=legend_elements,
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        title="Players"
    )

    plt.show()

In [58]:
# Slider
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(structure_df) - 1,
    step=1,
    description="Event:",
    continuous_update=False
)


output = widgets.interactive_output(
    plot_frame,
    {"frame": slider}
)


display(
    widgets.VBox([
        output,
        slider
    ])
)

# event 66 - 72
überlappung der fläche zwischen spieler um kontrollraum/heatmap dunkler darstellen 
